# Python Retail Sales Analysis

**Portfolio Project — Myiah Roseman**

This project analyzes a synthetic retail sales dataset using Python, pandas, and matplotlib.

## Business Questions

1. How much revenue and profit did the business generate?
2. Which months performed best?
3. Which categories and products drove the most revenue?
4. Which regions performed strongest?
5. What does customer repeat behavior look like?
6. What recommendations can be made from the data?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv("retail_sales.csv")
df.head()


## 1. Inspect and Clean the Data


In [ ]:
df.info()
df.isna().sum()


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["region"] = df["region"].fillna("Unknown")
df["customer_id"] = df["customer_id"].fillna("Guest")

df["calculated_revenue"] = (
    df["quantity"] * df["unit_price"] * (1 - df["discount_pct"])
).round(2)

df["calculated_profit"] = (
    df["calculated_revenue"] - (df["quantity"] * df["unit_cost"])
).round(2)

df.head()


## 2. KPI Summary


In [ ]:
total_revenue = df["revenue"].sum()
total_profit = df["profit"].sum()
average_order_value = df.groupby("order_id")["revenue"].sum().mean()
profit_margin = total_profit / total_revenue

{
    "Total Revenue": round(total_revenue, 2),
    "Total Profit": round(total_profit, 2),
    "Average Order Value": round(average_order_value, 2),
    "Profit Margin": round(profit_margin, 4),
}


## 3. Monthly Revenue Trend


In [ ]:
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_revenue = (
    df.groupby("month", as_index=False)["revenue"]
      .sum()
      .sort_values("month")
)

monthly_revenue


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(monthly_revenue["month"], monthly_revenue["revenue"], marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.show()


## 4. Category Performance


In [ ]:
category_performance = (
    df.groupby("category", as_index=False)
      .agg(
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units_sold=("quantity", "sum")
      )
)

category_performance["profit_margin"] = (
    category_performance["profit"] / category_performance["revenue"]
)

category_performance.sort_values("revenue", ascending=False)


In [ ]:
category_sorted = category_performance.sort_values("revenue", ascending=False)

plt.figure(figsize=(7, 5))
plt.bar(category_sorted["category"], category_sorted["revenue"])
plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.show()


## 5. Regional Sales


In [ ]:
regional_sales = (
    df.groupby("region", as_index=False)["revenue"]
      .sum()
      .sort_values("revenue", ascending=False)
)

regional_sales


## 6. Product Performance


In [ ]:
product_performance = (
    df.groupby("product", as_index=False)
      .agg(
          units_sold=("quantity", "sum"),
          revenue=("revenue", "sum"),
          profit=("profit", "sum")
      )
      .sort_values("revenue", ascending=False)
)

product_performance.head(10)


## 7. Customer Behavior


In [ ]:
customer_summary = (
    df.groupby("customer_id", as_index=False)
      .agg(
          orders=("order_id", "nunique"),
          revenue=("revenue", "sum")
      )
)

known_customers = customer_summary[customer_summary["customer_id"] != "Guest"]
repeat_customers = known_customers[known_customers["orders"] > 1]
repeat_customer_rate = len(repeat_customers) / len(known_customers)

print(f"Repeat customer rate: {repeat_customer_rate:.1%}")
known_customers.sort_values("revenue", ascending=False).head(10)


## 8. Key Findings

Based on the included synthetic dataset:

- **Total revenue:** $56,764.22
- **Total profit:** $28,122.22
- **Overall profit margin:** 49.5%
- **Average order value:** $87.33
- **Highest-revenue month:** 2025-11 ($6,535.76)
- **Top revenue category:** Electronics ($29,241.63)
- **Top region:** South ($18,552.51)
- **Top product:** Smart Watch ($12,277.51)
- **Repeat customer rate:** 80.3%

## Business Recommendations

1. Prioritize inventory and marketing around the strongest revenue categories and top-selling products.
2. Review seasonal patterns around the highest-performing months and plan promotions before demand peaks.
3. Investigate the strongest region for repeatable tactics that could be applied to weaker regions.
4. Use repeat-customer behavior to design retention campaigns and targeted offers.
5. Monitor discounting against profit margin so revenue growth does not come at the expense of profitability.
